# Experiment 003: 3B SFT-to-GRPO run, seed 2 (Colab)

One run: SFT-to-GRPO seed 2 under the unchanged frozen protocol, identical to
seeds 0 and 1 except the seed (pair-tested across all three configs). It
turns the 3B spread into a fully recorded three-seed observation, matching
0.5B and 7B. Base evals are reused from the shared 003 directory, so only
the SFT stage, the GRPO stage and the endpoint evals run.

Recorded environment must match the seed-1 sibling exactly; the deps cell
asserts it and stops the session otherwise. ~7h on an A100-80GB, one
session likely, resume-safe throughout.

If the run OOMs: do NOT rerun as-is. Archive the failed directory into
failed_attempts following the registered OOM procedure, then restart from
step zero.

## 0. Compute units and mount Drive

Fill `cu_before` and `unit_rate` now.

In [1]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 542.16    # from the resources panel
unit_rate = 542.16    # units per hour
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '003_colab_3b_seed2', 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260802_110314 | cu_before = 542.16
Mounted at /content/drive
closed previous session: 20260802_001712
saved open session: 20260802_110314
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: A100 with 80GB required

In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime for the 3b run, its peak was 50.4GB'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-46e1ee5a-9f1f-0465-53fd-a83f8457b7ed)
total vram: 79.3 GiB


## 2. Clone or update the repo (automatic)

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 3147, done.
remote: Counting objects: 100% (631/631), done.
remote: Compressing objects: 100% (404/404), done.
remote: Total 3147 (delta 405), reused 448 (delta 227), pack-reused 2516 (from 1)
Receiving objects: 100% (3147/3147), 87.46 MiB | 18.62 MiB/s, done.
Resolving deltas: 100% (2024/2024), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')

# match the recorded 003 seed-1 environment before training
# require the reference manifest so a missing record stops the session
import importlib, json, os
ref = 'results/003_COLAB_FULL_B3_sft_grpo_seed1/manifest.json'
assert os.path.exists(ref), 'reference manifest missing, pull the repo with results before training'
par = json.load(open(ref))['versions']
for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate'):
    have = importlib.import_module(lib).__version__
    assert have == par[lib], f'{lib} {have} differs from the recorded {par[lib]}'
print('environment matches the recorded 003 seed-1 versions')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 165.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.5 MB/s eta 0:00:00
deps installed
environment matches the recorded 003 seed-1 versions


## 4. W&B key (automatic)

Reminder: colab secrets are granted per notebook, switch on access for
`WANDB_API_KEY` under the key icon before running. This key is only used for experiment tracking.


In [5]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check the Drive record and free space

In [6]:
import os, shutil
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 20, 'need >=20GB free on drive for the 3b run'

open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260802_110314.json
drive free: 179 GB


## 6. Build the data splits

In [7]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 19.3MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:01<00:00, 1.65MB/s,  224kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:01<00:00, 1.65MB/s,  224kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:01<00:00, 321kB/s, 40.8kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:01<00:00, 321kB/s, 40.8kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 392502.02 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 266077.67 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 86.1MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 134MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00:00<?, ?B

## 7. Preflight: the shared 003 base evals must exist

In [8]:
import os
D = '/content/drive/MyDrive/inm363'
be = os.path.join(D, 'results', 'base_evals', '003_COLAB_FULL_B3')
for f in ('base_eval.json', 'base_logprob.json'):
    assert os.path.exists(os.path.join(be, f)), f'shared 003 base eval missing: {f}'
print('shared base evals present, the run will skip them')

shared base evals present, the run will skip them


## 8. Wiring check

In [9]:
!python scripts/run.py --config configs/003_colab_full_b3_seed2.yaml

env=COLAB mode=FULL scale=B3 condition=SFT_GRPO
model=Qwen/Qwen2.5-3B device=cuda stages=['base_eval', 'sft', 'sft_eval', 'grpo', 'grpo_eval']
steps=500 N=8 seed=2
trackers=['tensorboard', 'wandb']
results -> /content/drive/MyDrive/inm363/results/003_COLAB_FULL_B3_sft_grpo_seed2/  checkpoints -> /content/drive/MyDrive/inm363/checkpoints/003_COLAB_FULL_B3_sft_grpo_seed2/
local smoke wiring ok


## 9. The run

**If colab disconnects:** re-run this cell, finished stages skip.

In [10]:
# both cost fields must be filled before the long run starts
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
# python loop with ! calls so the logs stream live, exit code checked so a
# failed run stops the session instead of hiding behind the next one
for cfg in ('003_colab_full_b3_seed2',):
    print(f'===== {cfg} =====')
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/{cfg}.yaml --stage all
    if _exit_code != 0:
        raise SystemExit(f'{cfg} failed with exit code {_exit_code}, see the OOM protocol before retrying')

Streaming output truncated to the last 5000 lines.
│ │ squarely in   │               │               │              │           │ │
│ │ front of all  │               │               │              │           │ │
│ │ the other     │               │               │              │           │ │
│ │ kids to watch │               │               │              │           │ │
│ │ a magician.   │               │               │              │           │ │
│ │ Piaget would  │               │               │              │           │ │
│ │ attribute     │               │               │              │           │ │
│ │ this to       │               │               │              │           │ │
│ │ Kate's        │               │               │              │           │ │
│ │ A.            │               │               │              │           │ │
│ │ egocentrism.  │               │               │              │           │ │
│ │ B. lack of    │               │               │       

## 10. Peek at the results

In [11]:
import json, os
D = '/content/drive/MyDrive/inm363'
for tag in ('003_COLAB_FULL_B3_sft_grpo_seed0', '003_COLAB_FULL_B3_sft_grpo_seed1',
            '003_COLAB_FULL_B3_sft_grpo_seed2'):
    sm = os.path.join(D, 'results', tag, 'summary.json')
    if os.path.exists(sm):
        j = json.load(open(sm))
        print(f"{tag}: sft {j['sft_acc']}  grpo {j['grpo_acc']}")
    else:
        print(f'{tag}: no summary yet')

003_COLAB_FULL_B3_sft_grpo_seed0: sft {'easy': 0.608646188850967, 'medium': 0.4876847290640394, 'hard': 0.4393939393939394}  grpo {'easy': 0.8213879408418657, 'medium': 0.6912972085385879, 'hard': 0.5202020202020202}
003_COLAB_FULL_B3_sft_grpo_seed1: sft {'easy': 0.5870307167235495, 'medium': 0.44499178981937604, 'hard': 0.43686868686868685}  grpo {'easy': 0.8373151308304891, 'medium': 0.6814449917898193, 'hard': 0.5505050505050505}
003_COLAB_FULL_B3_sft_grpo_seed2: sft {'easy': 0.6245733788395904, 'medium': 0.5090311986863711, 'hard': 0.4772727272727273}  grpo {'easy': 0.8498293515358362, 'medium': 0.6863711001642037, 'hard': 0.5176767676767676}


## 11. Close the units record

In [13]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 6.77     # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

logged: {'session_id': '20260802_110314', 'ts_start': '2026-08-02 11:03', 'notebook': '003_colab_3b_seed2', 'colab_cu_before': 542.16, 'unit_rate': 542.16, 'ts_end': '2026-08-02 18:01', 'colab_cu_after': 6.77, 'colab_cu_used': 535.39, 'closed_on_resume': False}
